<a href="https://colab.research.google.com/github/Keerthanabs1326/Ethnotech_GenAI/blob/main/tacotron.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import numpy as np
import IPython.display as ipd
from scipy.io.wavfile import write

# Change to T4
# Tacotron2 and WaveGlow are very slow on CPU, so GPU is recommended.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load the pretrained Tacotron2 model from NVIDIA TorchHub.
# Tacotron2 converts input text into a Mel-spectrogram (audio feature representation).
tacotron2 = torch.hub.load(
    "nvidia/DeepLearningExamples:torchhub",
    "nvidia_tacotron2"
)

# Load the pretrained WaveGlow vocoder from TorchHub.
# WaveGlow converts the Mel-spectrogram into a real audio waveform (speech).
waveglow = torch.hub.load(
    "nvidia/DeepLearningExamples:torchhub",
    "nvidia_waveglow"
)

# Move both models completely to the selected device (GPU or CPU).
# This avoids device mismatch errors during inference.
tacotron2 = tacotron2.to(device)
waveglow = waveglow.to(device)

# Set both models to evaluation mode since we are not training them.
tacotron2.eval()
waveglow.eval()

print("Tacotron2 and WaveGlow models loaded successfully!")

# Tacotron2 needs text to be converted into numeric token IDs.
# This function applies text cleaning + converts characters into model input format.
from tacotron2.text import text_to_sequence

# This is the text that will be converted into speech.
text = "Hello class, How are you? Tacotron two is finally working perfectly, How are Gen AI classes are going?"

# Convert the text into a sequence of token IDs that Tacotron2 understands.
sequence = np.array(text_to_sequence(text, ['english_cleaners']))[None, :]
sequence = torch.LongTensor(sequence).to(device)

# Tacotron2 also requires the length of the input sequence.
# This helps the model know how many tokens are valid.
input_lengths = torch.LongTensor([sequence.shape[1]]).to(device)

# Run Tacotron2 inference:
# This generates a Mel-spectrogram representation of the spoken sentence.
with torch.no_grad():
    mel, _, _ = tacotron2.infer(sequence, input_lengths)

# Run WaveGlow inference:
# This converts the Mel-spectrogram into an actual speech waveform.
with torch.no_grad():
    audio = waveglow.infer(mel)

# Convert the output waveform tensor into a NumPy array for saving.
audio = audio[0].cpu().numpy()

# Save the generated speech into a WAV file.
write("speech.wav", 22050, audio)

print("Speech generated successfully! File saved as speech.wav")

# Play the generated audio directly inside the Colab notebook.
ipd.Audio("speech.wav")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/torch/hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/nvidia/DeepLearningExamples/zipball/torchhub" to /root/.cache/torch/hub/torchhub.zip


/root/.cache/torch/hub/nvidia_DeepLearningExamples_torchhub/PyTorch/Classification/ConvNets/image_classification/models/common.py:13: UserWarning: pytorch_quantization module not found, quantization will not be available
  warnings.warn(
/root/.cache/torch/hub/nvidia_DeepLearningExamples_torchhub/PyTorch/Classification/ConvNets/image_classification/models/efficientnet.py:17: UserWarning: pytorch_quantization module not found, quantization will not be available
  warnings.warn(
Using cache found in /root/.cache/torch/hub/nvidia_DeepLearningExamples_torchhub
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Tacotron2 and WaveGlow models loaded successfully!


/root/.cache/torch/hub/nvidia_DeepLearningExamples_torchhub/PyTorch/SpeechSynthesis/Tacotron2/tacotron2/text/__init__.py:74: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
  return s in _symbol_to_id and s is not '_' and s is not '~'
/root/.cache/torch/hub/nvidia_DeepLearningExamples_torchhub/PyTorch/SpeechSynthesis/Tacotron2/tacotron2/text/__init__.py:74: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
  return s in _symbol_to_id and s is not '_' and s is not '~'


Speech generated successfully! File saved as speech.wav
